In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df_list = []

file_name_list = [
    "DMEC_Deaths_1-1-21_to_12-31-2022_-_unedited.xlsx",
    "DMEC_Deaths_All_Cases_2022_and_2023_-_unedited.xlsx",
    "PRA_UCLA_DMEC_Deaths_2012_2021.xlsx",
    "PRA_DME_Cases_10-2023_to_7-2024_.csv",
    "UCLAAllClosedCases.csv",
    "UCLA_Romero_Ruby_Request_10.10.24.csv",
]

dmec_0121_1231 = pd.read_excel(
    "./pipeline_steps/input_files/raw/DMEC_Deaths_1-1-21_to_12-31-2022_-_unedited.xlsx"
)
df_list.append(dmec_0121_1231)

dmec_2022_2023 = pd.read_excel(
    "./pipeline_steps/input_files/raw/DMEC_Deaths_All_Cases_2022_and_2023_-_unedited.xlsx"
)
df_list.append(dmec_2022_2023)

demc_2012_2021 = pd.read_excel(
    "./pipeline_steps/input_files/raw/PRA_UCLA_DMEC_Deaths_2012_2021.xlsx"
)
df_list.append(demc_2012_2021)

dmec_1023_0724 = pd.read_csv(
    "./pipeline_steps/input_files/converted/PRA_DME_Cases_10-2023_to_7-2024_.csv"
)
df_list.append(dmec_1023_0724)

dmec_ucla_closed = pd.read_csv(
    "./pipeline_steps/input_files/converted/UCLAAllClosedCases.csv"
)
df_list.append(dmec_ucla_closed)

dmec_ucla_1024 = pd.read_csv(
    "./pipeline_steps/input_files/converted/UCLA_Romero_Ruby_Request_10.10.24.csv"
)
df_list.append(dmec_ucla_1024)

In [ ]:
for index, df in enumerate(df_list):
    try:
        df["DeathDate"]
    except:
        df = df.rename(columns={"Date of Death": "DeathDate"})
        df_list[index] = df

In [ ]:
from collections import defaultdict
from datetime import datetime


# Assuming your dataframes are in a list called `dataframes`
def group_by_monthly_coverage(dataframes):
    monthly_groups = defaultdict(list)

    for i, df in enumerate(dataframes):
        # Ensure DeathDate is a datetime object
        df["DeathDate"] = pd.to_datetime(df["DeathDate"], errors="coerce")

        # Extract unique (year, month) pairs
        unique_months = df["DeathDate"].dropna().dt.to_period("M").unique()

        # Assign the dataframe to its corresponding months
        for month in unique_months:
            monthly_groups[month].append(i)

    return monthly_groups


# Example analysis of overlaps
def analyze_overlaps(monthly_groups):
    overlaps = {month: len(files) for month, files in monthly_groups.items()}
    return overlaps


def filter_high_overlap_groups(monthly_groups, min_overlap=3):
    filtered_groups = {
        month: group
        for month, group in monthly_groups.items()
        if len(group) >= min_overlap
    }
    return filtered_groups


def get_dataframes_from_groups(high_overlap_groups, dataframes):
    grouped_dataframes = {}

    for month, indices in high_overlap_groups.items():
        # Collect dataframes corresponding to each group
        grouped_dataframes[month] = [dataframes[i] for i in indices]

    return grouped_dataframes


monthly_groups = group_by_monthly_coverage(df_list)
overlaps = analyze_overlaps(monthly_groups)
high_overlap_groups = filter_high_overlap_groups(monthly_groups, 3)
# Print results
print("Monthly Groups:")
for month, group in monthly_groups.items():
    print(f"{month}: {group} (Files covering this month)")

print("\nOverlap Analysis:")
for month, count in overlaps.items():
    print(f"{month}: {count} files overlap")

In [ ]:
high_overlap_groups

high_overlap_dfs = get_dataframes_from_groups(high_overlap_groups, df_list)

In [ ]:
stringified_groups = {
    str(month): indices for month, indices in high_overlap_groups.items()
}

In [ ]:
for month, indices in stringified_groups.items():
    print(f"Month: {month}")
    for idx in indices:
        print(f"DataFrame {idx} covers this month:")
        print(df_list[idx].head())

In [ ]:
def analyze_case_overlap(
    high_overlap_groups,
    dataframes,
    file_names,
    unique_id_column="CaseNumber",
    date_column="DeathDate",
):
    overlap_analysis = {}

    for month, indices in high_overlap_groups.items():
        all_cases = set()  # Store all unique cases in this group
        case_counts = {}  # Store contribution of each dataframe

        for i, idx in enumerate(indices):
            df = dataframes[idx]  # Retrieve the actual dataframe using the index

            # Filter the dataframe for the specific month
            month_filter = pd.Period(month, freq="M")
            filtered_df = df[df[date_column].dt.to_period("M") == month_filter]

            # Extract unique cases from the filtered dataframe
            cases = set(filtered_df[unique_id_column].dropna())

            # Calculate the contribution of new cases
            case_counts[file_names[idx]] = len(cases - all_cases)

            # Update the union of cases
            all_cases.update(cases)

        overlap_analysis[month] = {
            "total_unique_cases": len(all_cases),
            "case_contributions": case_counts,
        }

    return overlap_analysis


for index, df in enumerate(df_list):
    try:
        df["CaseNumber"]
    except:
        df = df.rename(columns={"CaseNum": "CaseNumber"})
        df_list[index] = df

# Example usage
unique_id_column = (
    "CaseNumber"  # Replace with the name of your unique case identifier column
)
overlap_analysis = analyze_case_overlap(
    high_overlap_groups, df_list, file_name_list, unique_id_column
)

# Display the results
for month, analysis in overlap_analysis.items():
    print(f"Month: {month}")
    print(f"  Total Unique Cases: {analysis['total_unique_cases']}")
    for df, count in analysis["case_contributions"].items():
        print(f"    {df} adds {count} new cases")

In [ ]:
# Convert overlap_analysis into a dataframe
def prepare_plot_data(overlap_analysis):
    plot_data = []
    for month, analysis in overlap_analysis.items():
        for file_name, count in analysis["case_contributions"].items():
            plot_data.append({"Month": month, "File": file_name, "Contribution": count})
    return pd.DataFrame(plot_data)


# Plot the contributions
def plot_contributions(data):
    # Ensure month order is chronological
    data["Month"] = data["Month"].apply(lambda x: pd.Period(x, freq="M").to_timestamp())
    data["Month"] = pd.to_datetime(
        data["Month"], format="%Y-%m"
    )  # Convert Month to datetime for sorting
    data = data.sort_values(by="Month")

    # Plot data
    pivot_data = data.pivot(index="Month", columns="File", values="Contribution")
    ax = pivot_data.plot(kind="bar", stacked=True, figsize=(12, 6))

    ax.set_xticks(range(len(pivot_data.index)))
    ax.set_xticklabels([d.strftime("%Y-%m") for d in pivot_data.index], rotation=45)

    plt.title("Contribution of Each File to Unique Cases per Month")
    plt.xlabel("Month")
    plt.ylabel("Number of New Cases")
    plt.legend(title="File", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_data = prepare_plot_data(overlap_analysis)
plot_contributions(plot_data)

In [ ]:
def create_overlap_analysis_table(overlap_analysis):
    rows = []
    for month, analysis in overlap_analysis.items():
        for file_name, count in analysis["case_contributions"].items():
            rows.append(
                {
                    "Month": str(month),
                    "File Name": file_name,
                    "Contribution": count,
                    "Total Unique Cases": analysis["total_unique_cases"],
                }
            )
    return pd.DataFrame(rows)


# Example: Convert the overlap analysis to a DataFrame
overlap_analysis_table = create_overlap_analysis_table(overlap_analysis)

In [ ]:
overlap_analysis_table["Month"] = pd.to_datetime(overlap_analysis_table["Month"])

In [ ]:
overlap_analysis_table = overlap_analysis_table.sort_values(by="Month")

In [ ]:
overlap_analysis_table.to_csv("./overlap_analysis_202309_202404.csv")

In [ ]:
def filter_dates_by_month(df, column, year, month):
    return df[(df[column].dt.year == year) & (df[column].dt.month == month)]


# Example usage: Filter for October 2023
filtered_dates = filter_dates_by_month(dmec_2022_2023, "DeathDate", 2023, 10)

In [ ]:
dmec_2022_2023